# ACA Medicaid Expansion and Uninsurance Convergence Across States
ECON 630: Week 6 Assignment
Julia van Helmond

## Setup: imports

`os` reads the environment variable, `Path` handles the cache folder, `requests` calls the Census Data API,
`pandas` holds the data, and `python-dotenv` loads the API key from `.env`.

In [1]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

## Setup: API key and cache folder

`load_dotenv()` copies the variables in `.env` into the environment, so the key is read without being
written in the notebook. The key is never printed; the cell only checks that it was found.

Each raw API response is saved to `data/raw/` (Plan §2). Later runs read those files instead of calling the API again,
which makes reruns fast and gives the same data every time.

In [2]:
load_dotenv()
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")
assert CENSUS_API_KEY, "CENSUS_API_KEY not found in .env"

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

## Helper: call the Census API

Both data sources use the same Census API format: the first row of the JSON is the column names and the
rest are data rows. This helper sends the request, turns the response into a DataFrame, and returns `None`
when the API has no data for that request (HTTP 204), so missing years can be skipped.

On an error it reports only the status code and message. It does not show the request URL, because the URL contains the key.

In [3]:
def census_get(url, params):
    """Call the Census API and return the response as a DataFrame (None if no data)."""
    resp = requests.get(url, params={**params, "key": CENSUS_API_KEY}, timeout=60)
    if resp.status_code == 204:
        return None
    if resp.status_code != 200:
        # Leave out resp.url so the key does not show up in the output
        raise RuntimeError(f"Census API error {resp.status_code}: {resp.text[:200]}")
    rows = resp.json()
    return pd.DataFrame(rows[1:], columns=rows[0])

## ACS 1-year pull (Plan §2a)

`fetch_acs_year(year)` requests the B27010 health-insurance counts and B19013 median household income for all
states in one year. If `data/raw/acs_{year}.csv` already exists, it reads that file instead. It adds a `year` column so the years
can be stacked. Puerto Rico (FIPS 72) is removed if it appears.

The years are 2010–2019 and 2021–2022. 2020 is skipped because the Census Bureau did not release standard 2020 ACS 1-year estimates.

In [4]:
ACS_URL = "https://api.census.gov/data/{year}/acs/acs1"
ACS_VARS = [
    "B27010_001E",                                # total civilian noninstitutionalized population
    "B27010_002E", "B27010_018E", "B27010_034E",  # population, three under-65 age groups
    "B27010_017E", "B27010_033E", "B27010_050E",  # no health insurance, three under-65 age groups
    "B27010_066E",                                # no health insurance, 65 and over
    "B19013_001E",                                # median household income
]
ACS_YEARS = list(range(2010, 2020)) + [2021, 2022]


def fetch_acs_year(year):
    """Return one year of ACS state data, read from the cache when it exists."""
    cache = RAW_DIR / f"acs_{year}.csv"
    if cache.exists():
        df = pd.read_csv(cache, dtype=str)
    else:
        params = {"get": ",".join(["NAME"] + ACS_VARS), "for": "state:*"}
        df = census_get(ACS_URL.format(year=year), params)
        df.to_csv(cache, index=False)
    df = df[df["state"] != "72"].copy()  # drop Puerto Rico if present
    df[ACS_VARS] = df[ACS_VARS].apply(pd.to_numeric)
    df["year"] = year
    return df

Loop over the 12 years and stack them into one state-year DataFrame. Then show the first rows and the
number of states in each year. Each year should have 51 (50 states + DC).

In [5]:
acs = pd.concat([fetch_acs_year(y) for y in ACS_YEARS], ignore_index=True)
acs.head()

,NAME,B27010_001E,B27010_002E,B27010_018E,B27010_034E,B27010_017E,B27010_033E,B27010_050E,B27010_066E,B19013_001E,state,year
0,Alabama,4702769,1133638,1060965,1870571,66958,310679,307036,1919,40474,01,2010
1,Alaska,689249,187984,161419,288016,22843,53131,59692,1174,64576,02,2010
2,Arizona,6310174,1629931,1442598,2362085,207967,414258,433169,9738,46789,04,2010
3,Arkansas,2867611,709374,641055,1112378,46495,223941,229191,986,38307,05,2010
4,California,36815569,9288691,9023508,14326399,832752,2888172,3032396,71593,57708,06,2010


In [6]:
acs.groupby("year").size().rename("rows")

year
2010    51
2011    51
2012    51
2013    51
2014    51
2015    51
2016    51
2017    51
2018    51
2019    51
2021    51
2022    51
Name: rows, dtype: int64

## SAHIE pull (Plan §2b, robustness check)

`fetch_sahie_year(year)` does the same for the Small Area Health Insurance Estimates. It gets the percent uninsured
(`PCTUI_PT`) for adults ages 18–64 (`AGECAT=1`) with income at or below 138% of poverty (`IPRCAT=3`). That is the group
Medicaid expansion targets. It also sets `SEXCAT=0` and `RACECAT=0` (all sexes, all races) so the API returns one row per state
instead of one row per demographic cell.

The plan says not to assume every year is available. A year with no data returns `None`, gets skipped, and the
cell after the loop prints the years that actually came back.

In [6]:
SAHIE_URL = "https://api.census.gov/data/timeseries/healthins/sahie"
SAHIE_YEARS = range(2010, 2023)


def fetch_sahie_year(year):
    """Return one year of SAHIE state data (<=138% FPL, ages 18-64), or None if unavailable."""
    cache = RAW_DIR / f"sahie_{year}.csv"
    if cache.exists():
        df = pd.read_csv(cache, dtype=str)
    else:
        params = {
            "get": "NAME,PCTUI_PT",
            "for": "state:*",
            "IPRCAT": 3,   # income <= 138% of poverty
            "AGECAT": 1,   # ages 18-64
            "SEXCAT": 0,   # both sexes
            "RACECAT": 0,  # all races
            "time": year,
        }
        df = census_get(SAHIE_URL, params)
        if df is None:
            return None
        df.to_csv(cache, index=False)
    df = df[df["state"] != "72"].copy()  # drop Puerto Rico if present
    df["PCTUI_PT"] = pd.to_numeric(df["PCTUI_PT"])
    df["year"] = year
    return df

Stack the years that returned data, print which years they were, and show the first rows and the row count per year.

In [7]:
sahie_frames = [fetch_sahie_year(y) for y in SAHIE_YEARS]
sahie = pd.concat([df for df in sahie_frames if df is not None], ignore_index=True)
print("SAHIE years returned:", sorted(sahie["year"].unique().tolist()))
sahie.head()

SAHIE years returned: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


,NAME,PCTUI_PT,IPRCAT,AGECAT,SEXCAT,RACECAT,time,state,year
0,Alabama,41.6,3,1,0,0,2010,01,2010
1,Alaska,47.7,3,1,0,0,2010,02,2010
2,Wyoming,42.7,3,1,0,0,2010,56,2010
3,Arizona,39.3,3,1,0,0,2010,04,2010
4,Arkansas,47.5,3,1,0,0,2010,05,2010


In [8]:
sahie.groupby("year").size().rename("rows")

year
2010    51
2011    51
2012    51
2013    51
2014    51
2015    51
2016    51
2017    51
2018    51
2019    51
2020    51
2021    51
2022    51
Name: rows, dtype: int64